# 🏠 Analyse et prédiction des prix immobiliers en Île-de-France (DVF)

Ce notebook présente les résultats du projet : exploration des données, analyses SQL, visualisations et modèle de prédiction.

**Prérequis** : avoir lancé `python run_all.py` au moins une fois (la base `db/dvf.db`, les exports et le modèle doivent exister).

In [ ]:
import sqlite3, json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.float_format", lambda v: f"{v:,.0f}".replace(",", " "))
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})

# Chemin du projet (à adapter si tu l'as déplacé)
ROOT = Path(r"C:\Users\HAYFA LAMOUCHI\Downloads\dvf-immobilier-idf\dvf-immobilier-idf")
if not (ROOT / "db" / "dvf.db").exists():      # sinon : dossier courant, puis son parent
    ROOT = Path.cwd()
    if not (ROOT / "db" / "dvf.db").exists():
        ROOT = ROOT.parent
DB = ROOT / "db" / "dvf.db"
assert DB.exists(), "Base introuvable : lance d'abord python run_all.py"
print("Projet :", ROOT)

## 1. Chargement des données depuis la base SQL

In [ ]:
QUERY = """
SELECT f.vente_id, d.date, d.annee, d.mois, d.annee_mois,
       l.code_commune, l.nom_commune, l.code_departement, t.type_local,
       f.valeur_fonciere, f.surface_reelle_bati, f.nombre_pieces, f.surface_terrain,
       f.prix_m2, f.latitude, f.longitude
FROM fact_ventes f
JOIN dim_date d         USING (date_id)
JOIN dim_localisation l USING (localisation_id)
JOIN dim_type_bien t    USING (type_bien_id)
"""
with sqlite3.connect(DB) as con:
    df = pd.read_sql_query(QUERY, con, parse_dates=["date"])

print(f"{len(df):,} ventes | {df['code_commune'].nunique()} communes | "
      f"du {df['date'].min():%d/%m/%Y} au {df['date'].max():%d/%m/%Y}".replace(",", " "))
df.head()

In [ ]:
df[["valeur_fonciere", "surface_reelle_bati", "nombre_pieces", "surface_terrain", "prix_m2"]].describe().T

## 2. Répartition des ventes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["type_local"].value_counts().plot.bar(ax=axes[0], color=["#0B5CAB", "#7FB2E5"], rot=0)
axes[0].set_title("Ventes par type de bien"); axes[0].set_xlabel("")
df["code_departement"].value_counts().sort_index().plot.bar(ax=axes[1], color="#0B5CAB", rot=0)
axes[1].set_title("Ventes par département"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()

## 3. Distribution du prix au m²

In [ ]:
fig, ax = plt.subplots()
for t, g in df.groupby("type_local"):
    ax.hist(g["prix_m2"], bins=80, alpha=0.6, label=t)
ax.set_xlabel("Prix au m² (€)"); ax.set_ylabel("Nombre de ventes"); ax.legend()
ax.set_title("Distribution du prix au m²")
plt.show()

## 4. Prix au m² médian par département (requête SQL)

In [ ]:
SQL_DEP = """
WITH ranked AS (
    SELECT l.code_departement, t.type_local, f.prix_m2,
           ROW_NUMBER() OVER (PARTITION BY l.code_departement, t.type_local ORDER BY f.prix_m2) AS rn,
           COUNT(*)     OVER (PARTITION BY l.code_departement, t.type_local) AS n
    FROM fact_ventes f
    JOIN dim_localisation l USING (localisation_id)
    JOIN dim_type_bien t    USING (type_bien_id)
)
SELECT code_departement, type_local, n AS nb_ventes, ROUND(AVG(prix_m2), 0) AS prix_m2_median
FROM ranked
WHERE rn IN ((n + 1) / 2, (n + 2) / 2)
GROUP BY code_departement, type_local, n
ORDER BY code_departement, type_local
"""
with sqlite3.connect(DB) as con:
    dep = pd.read_sql_query(SQL_DEP, con)

pivot = dep.pivot(index="code_departement", columns="type_local", values="prix_m2_median")
ax = pivot.plot.bar(rot=0, color=["#0B5CAB", "#7FB2E5"])
ax.set_title("Prix au m² médian par département"); ax.set_xlabel("Département"); ax.set_ylabel("€ / m²")
plt.show()
dep

## 5. Évolution des prix dans le temps

In [ ]:
evo = (df.groupby(["annee_mois", "type_local"])["prix_m2"].median().unstack())
ax = evo.plot(marker="o", markersize=3, color=["#0B5CAB", "#E07B39"])
ax.set_title("Prix au m² médian par mois"); ax.set_xlabel(""); ax.set_ylabel("€ / m²")
plt.xticks(rotation=45); plt.show()

vol = df.groupby("annee_mois").size()
ax = vol.plot.bar(color="#0B5CAB", figsize=(11, 3.5))
ax.set_title("Nombre de ventes par mois"); ax.set_xlabel("")
ax.set_xticks(ax.get_xticks()[::3]); plt.show()

## 6. Communes les plus chères et les moins chères (au moins 30 ventes)

In [ ]:
com = (df.groupby(["code_commune", "nom_commune", "code_departement"])
         .agg(nb_ventes=("prix_m2", "size"), prix_m2_median=("prix_m2", "median"))
         .reset_index())
com = com[com["nb_ventes"] >= 30].sort_values("prix_m2_median", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
top = com.head(15).iloc[::-1]
axes[0].barh(top["nom_commune"], top["prix_m2_median"], color="#0B5CAB")
axes[0].set_title("15 communes les plus chères (€/m² médian)")
bot = com.tail(15).iloc[::-1]
axes[1].barh(bot["nom_commune"], bot["prix_m2_median"], color="#7FB2E5")
axes[1].set_title("15 communes les moins chères (€/m² médian)")
plt.tight_layout(); plt.show()

## 7. Carte des prix

In [ ]:
s = df.sample(min(40_000, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(s["longitude"], s["latitude"], c=s["prix_m2"], s=3, cmap="viridis",
                vmin=s["prix_m2"].quantile(0.02), vmax=s["prix_m2"].quantile(0.98))
plt.colorbar(sc, label="Prix au m² (€)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Prix au m² des ventes (échantillon)")
ax.set_aspect(1 / np.cos(np.radians(48.8)))
plt.show()

## 8. Modèle de prédiction : performances

In [ ]:
metrics = json.loads((ROOT / "reports" / "metrics.json").read_text(encoding="utf-8"))
res = pd.DataFrame(metrics["resultats"]).T
res["R2"] = res["R2"].map(lambda v: f"{v:.3f}")
res["Erreur relative médiane (%)"] = res["Erreur relative médiane (%)"].map(lambda v: f"{v:.1f}")
print(f"Entraînement : {metrics['n_train']:,} ventes | Test : {metrics['n_test']:,} ventes".replace(",", " "))
print("Meilleur modèle :", metrics["meilleur_modele"])
res

In [ ]:
display(Image(filename=str(ROOT / "reports" / "figures" / "predit_vs_reel.png"), width=480))
display(Image(filename=str(ROOT / "reports" / "figures" / "importance_variables.png"), width=560))

## 9. Où le modèle se trompe-t-il ?

In [ ]:
pred = pd.read_csv(ROOT / "exports" / "powerbi" / "predictions_test.csv",
                   dtype={"code_departement": str})
pred["erreur_abs_pct"] = pred["erreur_pct"].abs()

print(f"Erreur relative médiane : {pred['erreur_abs_pct'].median():.1f} %")
print(f"Part des prédictions à moins de 10 % du prix réel : {(pred['erreur_abs_pct'] < 10).mean():.0%}")
print(f"Part des prédictions à moins de 20 % du prix réel : {(pred['erreur_abs_pct'] < 20).mean():.0%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pred.groupby("code_departement")["erreur_abs_pct"].median().plot.bar(ax=axes[0], color="#0B5CAB", rot=0)
axes[0].set_title("Erreur relative médiane par département (%)"); axes[0].set_xlabel("")
pred.groupby("type_local")["erreur_abs_pct"].median().plot.bar(ax=axes[1], color="#7FB2E5", rot=0)
axes[1].set_title("Erreur relative médiane par type de bien (%)"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()

## 10. Essayer le modèle sur un bien

In [ ]:
model = joblib.load(ROOT / "models" / "meilleur_modele.joblib")

# Modifie les valeurs ci-dessous pour tester d'autres biens
bien = pd.DataFrame([{
    "surface_reelle_bati": 65,
    "nombre_pieces": 3,
    "surface_terrain": 0,
    "latitude": 48.86,
    "longitude": 2.35,
    "annee": int(df["annee"].max()),
    "mois": 6,
    "type_local": "Appartement",
    "code_departement": "75",
}])
prix = model.predict(bien)[0]
print(f"Prix estimé : {prix:,.0f} €  ({prix / bien.loc[0, 'surface_reelle_bati']:,.0f} €/m²)".replace(",", " "))

## 11. Conclusions

*À compléter avec tes propres observations après avoir regardé les graphiques, par exemple :*

- Quels départements et communes sont les plus chers ? les moins chers ?
- Comment le prix au m² a-t-il évolué sur la période ? Y a-t-il une différence entre appartements et maisons ?
- Quel modèle est le meilleur et avec quelle précision ?
- Quelles variables comptent le plus dans la prédiction ?
- Limites : pas d'information sur l'étage, l'état du bien ou le DPE ; découpage entraînement/test aléatoire.